# Week 60, Day 2: Gaussian mixtures and the EM algorithm

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w060/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: EM with full covariances

Complete the E step (responsibilities from the weighted Gaussian densities) and the M step (weights, means and covariances from the responsibilities) of gmm_em, then fit three components to 600 points from a mixture of elongated, tilted Gaussians.

In [ ]:
import numpy as np
from scipy.stats import multivariate_normal as mvn

def gmm_em(X, K, g, iters=200, reg=1e-6):
    n, d = X.shape
    mu = X[g.choice(n, K, replace=False)]
    Sig = np.array([np.cov(X.T) for _ in range(K)]); pi = np.full(K, 1 / K)
    ll_hist = []
    for _ in range(iters):
        dens = np.column_stack([pi[k] * mvn.pdf(X, mu[k], Sig[k]) for k in range(K)])
        ll_hist.append(np.log(dens.sum(1)).sum())
        R = ...                                                  # E step
        Nk = R.sum(0)
        pi = ...                                                 # M step
        mu = (R.T @ X) / Nk[:, None]
        for k in range(K):
            D = X - mu[k]
            Sig[k] = ... + reg * np.eye(d)
        if len(ll_hist) > 1 and ll_hist[-1] - ll_hist[-2] < 1e-8:
            break
    return pi, mu, Sig, R, ll_hist

ge = np.random.default_rng(602)
def tilted(n, m, sx, sy, ang, g):
    Rm = np.array([[np.cos(ang), -np.sin(ang)], [np.sin(ang), np.cos(ang)]])
    return m + (g.normal(size=(n, 2)) * [sx, sy]) @ Rm.T
Xg = np.vstack([tilted(300, [0, 0], 2.0, 0.4, 0.6, ge), tilted(180, [4, -1], 0.5, 1.5, 0.0, ge), tilted(120, [1, 4], 1.0, 0.3, -0.8, ge)])
truth_g = np.repeat([0, 1, 2], [300, 180, 120])
best = max((gmm_em(Xg, 3, ge) for _ in range(5)), key=lambda f: f[4][-1])
pi_hat, mu_hat, Sig_hat, R, ll_hist = best
print(f"{len(ll_hist)} iterations; log-likelihood {ll_hist[0]:.1f} -> {ll_hist[-1]:.1f}")
print("weights:", np.round(pi_hat, 3)); print("means:", np.round(mu_hat, 2).tolist())

In [ ]:
import numpy as np
def test_resp():
    "responsibilities are non-negative and sum to one for each point"
    assert np.all(R >= 0) and np.allclose(R.sum(1), 1)
def test_monotone():
    "the log-likelihood never decreases"
    assert all(b >= a - 1e-8 for a, b in zip(ll_hist, ll_hist[1:]))
def test_fit():
    "the fitted weights are close to the true 0.5, 0.3 and 0.2"
    assert np.allclose(np.sort(pi_hat), [0.2, 0.3, 0.5], atol=0.06)

test_resp()
test_monotone()
test_fit()
print("All checks passed")

### Exercise: Elongated clusters: k-means against a mixture

Assign each point to its most responsible component, and compare the agreement with the true groups against k-means (k = 3, best of 10 starts) on the same data.

In [ ]:
def kmeans_simple(X, k, g, iters=100):
    C = X[g.choice(len(X), k, replace=False)]
    for _ in range(iters):
        lab = np.argmin(((X[:, None] - C[None]) ** 2).sum(-1), 1)
        C = np.array([X[lab == j].mean(0) if np.any(lab == j) else C[j] for j in range(k)])
    return lab, ((X - C[lab]) ** 2).sum()

def agree(lab, truth, k=3):
    from itertools import permutations
    return max(np.mean(np.array(p)[lab] == truth) for p in permutations(range(k)))

lab_gmm = ...
acc_gmm = agree(lab_gmm, truth_g)
lab_km, _ = min((kmeans_simple(Xg, 3, ge) for _ in range(10)), key=lambda f: f[1])
acc_kmeans = agree(lab_km, truth_g)
print(f"agreement with true groups: mixture {acc_gmm:.3f}, k-means {acc_kmeans:.3f}")

In [ ]:
import numpy as np
def test_compare():
    "the mixture recovers the three tilted groups with accuracy above 0.9, and beats k-means"
    assert acc_gmm > 0.9 and acc_gmm > acc_kmeans + 0.05

test_compare()
print("All checks passed")

### Exercise: A component that collapses

Evaluate the log-likelihood of a three-component fit to Xg in which a fourth component, with weight 0.01 and covariance 1e-20 times the identity, sits on a single data point (the other three keep their parameters, with weights scaled by 0.99), and compare it with the honest fit. Then rerun EM on a small sample (20 points, 6 components) with reg = 1e-2 and record the smallest determinant of the fitted covariances.

In [ ]:
def mixture_ll(X, pi, mu, Sig):
    return np.log(sum(pi[k] * mvn.pdf(X, mu[k], Sig[k]) for k in range(len(pi)))).sum()

ll_honest = mixture_ll(Xg, pi_hat, mu_hat, Sig_hat)
pi_c = np.r_[0.99 * pi_hat, 0.01]
mu_c = np.vstack([mu_hat, Xg[0]])
Sig_c = np.concatenate([Sig_hat, [1e-20 * np.eye(2)]])
ll_collapsed = mixture_ll(Xg, pi_c, mu_c, Sig_c)
print(f"honest fit {ll_honest:.1f}; with a collapsed component {ll_collapsed:.1f}")

small = Xg[ge.choice(len(Xg), 20, replace=False)]
fit_reg = gmm_em(small, 6, np.random.default_rng(3), reg=...)
min_det_reg = min(np.linalg.det(S) for S in fit_reg[2])
print(f"20 points, 6 components, regularised: smallest covariance determinant {min_det_reg:.4f}")

In [ ]:
import numpy as np
def test_blowup():
    "without regularisation, the log-likelihood with a collapsed component exceeds the honest fit by more than 20"
    assert ll_collapsed > ll_honest + 20
def test_reg():
    "with regularisation, no covariance determinant falls below reg squared (1e-4)"
    assert min_det_reg > 0.9e-4

test_blowup()
test_reg()
print("All checks passed")